# Stage 1: Inhibitory Scaffold
## Ascoli Framework — First-Order Inhibitory Modulation
**Hippocampal Connectome Analysis | Ascoli Lab, GMU**

This notebook builds directly on Stage 0. The E→E greedy path is **not recomputed** — it is loaded from Stage 0's saved output. Inhibitory neurons enter as a **modulation layer annotated on top**, not as path redirectors.

**Scope:**
- Load Stage 0 path results and reload W, ei_df from source
- Single-step E→I→E sidechain annotation for each node on the canonical greedy path
- Terminus comparison: does inhibitory modulation reroute the greedy path?
- Full E→I→E→I chaining — Variant A (stop), Variant B (skip), Variant C (edge exhaustion)
- Visualizations

**Key design decisions:**
- E→E greedy path is preserved unchanged (pure annotation approach)
- Suppression target lookup restricted to excitatory nodes only
- 'Stages' not 'layers' (per Dr. Ascoli's convention)

## 0. Imports & Configuration

Same dual-environment setup as Stage 0. Google Colab is the primary working environment; local execution is supported for the PI's convenience when the notebook is downloaded with the data files in the standard folder layout.

**Expected layout:**
```
Ascoli_LabRotation/
  matrices/
    w_ij_gaa.csv
    izhikevich_72_v3.csv
  outputs/
    scaffolds/          <- Stage 0 outputs loaded from here
    stage1/             <- Stage 1 outputs written here
  stage1_inhibitory_scaffold_CLEAN.ipynb
```

In [1]:
import sys
import os
import warnings
from collections import defaultdict

# ── Environment detection & setup ─────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Ascoli_LabRotation/'
    print('Colab environment detected. Drive mounted.')
else:
    BASE = ''
    print(f'Local environment detected. CWD = {os.getcwd()}')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

# ── File paths ────────────────────────────────────────────────────────────────
WEIGHT_MATRIX  = os.path.join(BASE, 'matrices', 'w_ij_gaa.csv')
NEURON_PARAMS  = os.path.join(BASE, 'matrices', 'izhikevich_72_v3.csv')
STAGE0_DIR     = os.path.join(BASE, 'outputs', 'scaffolds', '')
TABLE_DIR      = os.path.join(BASE, 'outputs', 'stage1', '')
os.makedirs(TABLE_DIR, exist_ok=True)

# Stage 0 CSV outputs loaded by this notebook
STAGE0_PATHS   = os.path.join(STAGE0_DIR, 'stage0_variantA_paths.csv')
STAGE0_RANKS   = os.path.join(STAGE0_DIR, 'stage0_hub_rankings.csv')
STAGE0_EE_NET  = os.path.join(STAGE0_DIR, 'ee_netlist.csv')

# ── Region colors (canonical across all stages) ───────────────────────────────
region_colors = {
    'DG': '#E67E22', 'CA3': '#2ECC71', 'CA2': '#1ABC9C', 'CA1': '#3498DB',
    'MEC': '#9B59B6', 'LEC': '#E91E63', 'EC': '#95A5A6', 'Other': '#7F8C8D',
}
region_order_list = ['DG', 'CA3', 'CA2', 'CA1', 'MEC', 'LEC', 'EC', 'Other']

# ── MEC naming patch (must be applied before any node list construction) ──────
MEC_PATCH = {'MEC LV Multipolar Pyramidal': 'MEC LIII Multipolar Principal'}

def region(name):
    for r in ['DG', 'CA3', 'CA2', 'CA1', 'MEC', 'LEC', 'EC']:
        if name.startswith(r): return r
    return 'Other'

# ── File check ────────────────────────────────────────────────────────────────
files = [
    ('Weight matrix',     WEIGHT_MATRIX),
    ('Neuron params',     NEURON_PARAMS),
    ('Stage 0 paths',     STAGE0_PATHS),
    ('Stage 0 rankings',  STAGE0_RANKS),
    ('Stage 0 EE netlist',STAGE0_EE_NET),
]
for label, path in files:
    status = '✓' if os.path.exists(path) else '✗ NOT FOUND'
    print(f'  {label:<22}: {status}  ({path})')
print(f'\nOutput directory: {TABLE_DIR}')

Mounted at /content/drive
Colab environment detected. Drive mounted.
  Weight matrix         : ✓  (/content/drive/MyDrive/Ascoli_LabRotation/matrices/w_ij_gaa.csv)
  Neuron params         : ✓  (/content/drive/MyDrive/Ascoli_LabRotation/matrices/izhikevich_72_v3.csv)
  Stage 0 paths         : ✓  (/content/drive/MyDrive/Ascoli_LabRotation/outputs/scaffolds/stage0_variantA_paths.csv)
  Stage 0 rankings      : ✓  (/content/drive/MyDrive/Ascoli_LabRotation/outputs/scaffolds/stage0_hub_rankings.csv)
  Stage 0 EE netlist    : ✓  (/content/drive/MyDrive/Ascoli_LabRotation/outputs/scaffolds/ee_netlist.csv)

Output directory: /content/drive/MyDrive/Ascoli_LabRotation/outputs/stage1/


## 1. Load Data

W and ei_df are reloaded from source files — Stage 1 is self-contained and does not depend on Stage 0 being in memory.

Stage 0 CSV outputs (path table, hub rankings, EE netlist) are loaded to reconstruct the canonical greedy path without rerunning the Stage 0 notebook. The MEC naming patch is applied to W immediately after loading.

In [2]:
# ── Reload weight matrix and neuron params from source ───────────────────────
W_raw = pd.read_csv(WEIGHT_MATRIX, index_col=0)
ei_df = pd.read_csv(NEURON_PARAMS)

# Apply MEC naming patch immediately — before any node list construction
W_raw.index   = [MEC_PATCH.get(n, n) for n in W_raw.index]
W_raw.columns = [MEC_PATCH.get(n, n) for n in W_raw.columns]
W = W_raw.copy()

w_nodes   = list(W.index)
ei_lookup = dict(zip(ei_df['Neuron Type'], ei_df['E/I']))
excitatory = [n for n in ei_df[ei_df['E/I'] == 'e']['Neuron Type'].tolist() if n in w_nodes]
inhibitory = [n for n in ei_df[ei_df['E/I'] == 'i']['Neuron Type'].tolist() if n in w_nodes]

print(f'Weight matrix : {W.shape[0]}×{W.shape[1]}')
print(f'Excitatory    : {len(excitatory)}')
print(f'Inhibitory    : {len(inhibitory)}')

# ── Load Stage 0 outputs ──────────────────────────────────────────────────────
df_paths  = pd.read_csv(STAGE0_PATHS)
rank_df   = pd.read_csv(STAGE0_RANKS, index_col=0)
ee_df     = pd.read_csv(STAGE0_EE_NET)

# Reconstruct the canonical greedy path (Variant A from MEC LII Stellate —
# the top-ranked starting node by weight sum)
CANONICAL_START = 'MEC LII Stellate'
canonical_path_df = df_paths[
    df_paths['path_start'] == CANONICAL_START
].sort_values('step').reset_index(drop=True)
canonical_path = list(canonical_path_df['e_node'])

print(f'\nCanonical path loaded: {len(canonical_path)} nodes')
print(f'  Start   : {canonical_path[0]}')
print(f'  Terminus: {canonical_path[-1]}')
print(f'  Path    : {" → ".join(canonical_path)}')

Weight matrix : 72×72
Excitatory    : 28
Inhibitory    : 44

Canonical path loaded: 16 nodes
  Start   : MEC LII Stellate
  Terminus: DG Semilunar Granule
  Path    : MEC LII Stellate → CA3 Pyramidal → CA3c Pyramidal → CA1 Pyramidal → EC LV Deep Pyramidal → LEC LVI Multipolar Pyramidal → LEC LIII Multipolar Principal → LEC LIII Complex Pyramidal → EC LIII V Bipolar Pyramidal → EC LIV VI Deep Multipolar Principal → MEC LIII Multipolar Principal → MEC LV Superficial Pyramidal → MEC LV Pyramidal → DG Granule → DG Mossy → DG Semilunar Granule


## 2. Single-Step E→I→E Sidechain Annotation

For each excitatory node on the canonical greedy path:

1. **Find the strongest E→I target** — the inhibitory neuron receiving the highest-weight signal from this E node (`max(W[e_node, inhibitory])` where weight > 0)
2. **Find what that I neuron most suppresses** — the excitatory node receiving the most negative I→E weight (`min(W[i_node, excitatory])` where weight < 0)

This is a pure annotation — the E→E path is not altered. The sidechain shows the first-order inhibitory consequence of each step.

**Bug fixed:** Suppression target lookup is restricted to `excitatory` nodes only. An earlier version searched all 72 nodes, which produced nonsensical results where inhibitory neurons appeared to 'suppress' other inhibitory neurons.

In [3]:
def inhibitory_sidechain(e_node, W, inhibitory, excitatory):
    """For an E node: find strongest E->I target; find what that I node most suppresses.
    Suppression lookup restricted to excitatory nodes only.
    Returns a dict or None if no E->I edges exist.
    """
    e_to_i = W.loc[e_node, inhibitory]
    valid   = e_to_i[e_to_i > 0]
    if valid.empty:
        return None
    top_i   = valid.idxmax()
    top_i_w = valid.max()
    # Restrict to excitatory targets only
    i_to_e       = W.loc[top_i, excitatory]
    neg          = i_to_e[i_to_e < 0]
    suppressed   = neg.idxmin() if not neg.empty else None
    suppress_w   = neg.min()    if not neg.empty else None
    return {
        'top_i_neuron':    top_i,
        'top_i_weight':    top_i_w,
        'most_suppressed': suppressed,
        'suppress_weight': suppress_w,
    }


# ── Build sidechain annotation table ─────────────────────────────────────────
sidechain_records = []
for _, row in canonical_path_df.iterrows():
    sc = inhibitory_sidechain(row['e_node'], W, inhibitory, excitatory)
    sidechain_records.append({
        'step':            int(row['step']),
        'e_node':          row['e_node'],
        'region':          region(row['e_node']),
        'edge_weight':     row['edge_weight'],
        'top_i_neuron':    sc['top_i_neuron']    if sc else '',
        'top_i_weight':    sc['top_i_weight']    if sc else np.nan,
        'most_suppressed': sc['most_suppressed'] if sc else '',
        'suppress_weight': sc['suppress_weight'] if sc else np.nan,
    })

sc_df = pd.DataFrame(sidechain_records)
sc_df.to_csv(f'{TABLE_DIR}stage1_sidechain_annotation.csv', index=False)

print(f'PATH: {CANONICAL_START}')
print(f'{"Step":<5} {"E Node":<38} {"->E wt":>12} | {"Top I neuron":<33} {"E->I wt":>10}  ->suppresses->  {"Suppressed E node":<38} {"I->E wt":>10}')
print('-' * 135)
for _, row in sc_df.iterrows():
    ew  = f'->[{row["edge_weight"]:.2e}]' if pd.notna(row['edge_weight']) else '(start)  '
    ti  = row['top_i_neuron']    if row['top_i_neuron']    else '-'
    tiw = f'{row["top_i_weight"]:.2e}' if pd.notna(row['top_i_weight']) else ''
    ms  = row['most_suppressed'] if row['most_suppressed'] else '-'
    sw  = f'{row["suppress_weight"]:.2e}' if pd.notna(row['suppress_weight']) else ''
    print(f'{int(row["step"]):<5} {row["e_node"]:<38} {ew:>12} | {ti:<33} {tiw:>10}  {ms:<38}  {sw:>10}')
print(f'\nSaved: {TABLE_DIR}stage1_sidechain_annotation.csv')

PATH: MEC LII Stellate
Step  E Node                                       ->E wt | Top I neuron                         E->I wt  ->suppresses->  Suppressed E node                         I->E wt
---------------------------------------------------------------------------------------------------------------------------------------
0     MEC LII Stellate                          (start)   | CA3 Basket                          1.75e+05  CA3 Pyramidal                            -1.37e+04
1     CA3 Pyramidal                          ->[2.73e+05] | CA1 Recurrent O LM                  2.08e+06  CA1 Pyramidal                            -1.77e+02
2     CA3c Pyramidal                         ->[3.96e+05] | CA1 OR LM                           2.98e+06  CA1 Radiatum Giant                       -3.26e+02
3     CA1 Pyramidal                          ->[2.95e+06] | CA1 Recurrent O LM                  1.73e+05  CA1 Pyramidal                            -1.77e+02
4     EC LV Deep Pyramidal               

## 3. Full E→I→E→I Chaining — Three Variants

The single-step sidechain in Section 2 stops after one E→I→E hop. This section extends chaining to its natural limit, in three variants that mirror the logic of Stage 0's greedy path variants.

At each node the **largest absolute weight** is followed — because inhibitory weights are negative, `abs(weight)` selects the dominant connection regardless of sign. At E nodes this picks the strongest E→I target; at I nodes it picks the most suppressive I→E target (most negative = largest absolute value).

**Variant A (Stop on revisit):** Follow the highest-|w| alternating edge at each step. Stop immediately when the next node has already appeared in the chain. Produces the shortest chains — conservative and directly analogous to Stage 0 Variant A.

**Variant B (Skip revisit, continue):** When the best next step would revisit a node, skip it, record it as a side loop, and continue to the next best available unvisited target. Stops only when no unvisited targets remain. Produces longer chains that reveal more of the inhibitory network structure — directly analogous to Stage 0 Variant B.

**Variant C (Edge exhaustion, revisits allowed):** Each directed edge may be used at most once; nodes may be revisited freely. Stops when no unused outgoing edges remain from the current node. The most exhaustive variant — analogous to Stage 0 Variant C.

Each variant prints the **full path** for all 27 starting nodes, followed by a terminus distribution table. The terminus of each inhibitory chain is compared against the pure excitatory greedy terminus from Stage 0 (loaded from `stage0_variantA_paths.csv`) to answer the core research question: **does the inhibitory chain converge to the same attractor as the pure excitatory path, or does it get rerouted?**

In [5]:
import sys
import os
import warnings
from collections import defaultdict

# ── Environment detection & setup ──────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Ascoli_LabRotation/'
    print('Colab environment detected. Drive mounted.')
else:
    BASE = ''
    print(f'Local environment detected. CWD = {os.getcwd()}')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

# ── File paths ───────────────────────────────────────────
WEIGHT_MATRIX  = os.path.join(BASE, 'matrices', 'w_ij_gaa.csv')
NEURON_PARAMS  = os.path.join(BASE, 'matrices', 'izhikevich_72_v3.csv')
STAGE0_DIR     = os.path.join(BASE, 'outputs', 'scaffolds', '')
TABLE_DIR      = os.path.join(BASE, 'outputs', 'stage1', '')
os.makedirs(TABLE_DIR, exist_ok=True)

# Stage 0 CSV outputs loaded by this notebook
STAGE0_PATHS   = os.path.join(STAGE0_DIR, 'stage0_variantA_paths.csv')
STAGE0_RANKS   = os.path.join(STAGE0_DIR, 'stage0_hub_rankings.csv')
STAGE0_EE_NET  = os.path.join(STAGE0_DIR, 'ee_netlist.csv')

# ── Region colors (canonical across all stages) ───────────
region_colors = {
    'DG': '#E67E22', 'CA3': '#2ECC71', 'CA2': '#1ABC9C', 'CA1': '#3498DB',
    'MEC': '#9B59B6', 'LEC': '#E91E63', 'EC': '#95A5A6', 'Other': '#7F8C8D',
}
region_order_list = ['DG', 'CA3', 'CA2', 'CA1', 'MEC', 'LEC', 'EC', 'Other']

# ── MEC naming patch (must be applied before any node list construction) ──
MEC_PATCH = {'MEC LV Multipolar Pyramidal': 'MEC LIII Multipolar Principal'}

def region(name):
    for r in ['DG', 'CA3', 'CA2', 'CA1', 'MEC', 'LEC', 'EC']:
        if name.startswith(r): return r
    return 'Other'

# ── File check ─────────────────────────────────────
files = [
    ('Weight matrix',     WEIGHT_MATRIX),
    ('Neuron params',     NEURON_PARAMS),
    ('Stage 0 paths',     STAGE0_PATHS),
    ('Stage 0 rankings',  STAGE0_RANKS),
    ('Stage 0 EE netlist',STAGE0_EE_NET),
]
for label, path in files:
    status = '✓' if os.path.exists(path) else '✗ NOT FOUND'
    print(f'  {label:<22}: {status}  ({path})')
print(f'\nOutput directory: {TABLE_DIR}')


# ── Reload weight matrix and neuron params from source ───────────────────
W_raw = pd.read_csv(WEIGHT_MATRIX, index_col=0)
ei_df = pd.read_csv(NEURON_PARAMS)

# Apply MEC naming patch immediately ─ before any node list construction
W_raw.index   = [MEC_PATCH.get(n, n) for n in W_raw.index]
W_raw.columns = [MEC_PATCH.get(n, n) for n in W_raw.columns]
W = W_raw.copy()

w_nodes   = list(W.index)
ei_lookup = dict(zip(ei_df['Neuron Type'], ei_df['E/I']))
excitatory = [n for n in ei_df[ei_df['E/I'] == 'e']['Neuron Type'].tolist() if n in w_nodes]
inhibitory = [n for n in ei_df[ei_df['E/I'] == 'i']['Neuron Type'].tolist() if n in w_nodes]

# ── Load Stage 0 outputs ───────────────────────────────
df_paths  = pd.read_csv(STAGE0_PATHS)
rank_df   = pd.read_csv(STAGE0_RANKS, index_col=0)
ee_df     = pd.read_csv(STAGE0_EE_NET)

# Reconstruct the canonical greedy path (Variant A from MEC LII Stellate ──
# the top-ranked starting node by weight sum)
CANONICAL_START = 'MEC LII Stellate'
canonical_path_df = df_paths[
    df_paths['path_start'] == CANONICAL_START
].sort_values('step').reset_index(drop=True)
canonical_path = list(canonical_path_df['e_node'])


def inhibitory_sidechain(e_node, W, inhibitory, excitatory):
    """For an E node: find strongest E->I target; find what that I node most suppresses.
    Suppression lookup restricted to excitatory nodes only.
    Returns a dict or None if no E->I edges exist.
    """
    e_to_i = W.loc[e_node, inhibitory]
    valid   = e_to_i[e_to_i > 0]
    if valid.empty:
        return None
    top_i   = valid.idxmax()
    top_i_w = valid.max()
    # Restrict to excitatory targets only
    i_to_e       = W.loc[top_i, excitatory]
    neg          = i_to_e[i_to_e < 0]
    suppressed   = neg.idxmin() if not neg.empty else None
    suppress_w   = neg.min()    if not neg.empty else None
    return {
        'top_i_neuron':    top_i,
        'top_i_weight':    top_i_w,
        'most_suppressed': suppressed,
        'suppress_weight': suppress_w,
    }


# ── Build sidechain annotation table ─────────────────────────────
sidechain_records = []
for _, row in canonical_path_df.iterrows():
    sc = inhibitory_sidechain(row['e_node'], W, inhibitory, excitatory)
    sidechain_records.append({
        'step':            int(row['step']),
        'e_node':          row['e_node'],
        'region':          region(row['e_node']),
        'edge_weight':     row['edge_weight'],
        'top_i_neuron':    sc['top_i_neuron']    if sc else '',
        'top_i_weight':    sc['top_i_weight']    if sc else np.nan,
        'most_suppressed': sc['most_suppressed'] if sc else '',
        'suppress_weight': sc['suppress_weight'] if sc else np.nan,
    })

sc_df = pd.DataFrame(sidechain_records)




# ── Variant A: stop on revisit ──────────────────────────────────
def ei_chain_varA(start_e, W, excitatory, inhibitory, max_steps=100):
    """E→I→E→I chain, stop when next node already visited.
    Follows largest abs(weight) at each step.
    Returns (chain, side_loops=[], reason).
    """
    chain   = [(start_e, 'E')]
    visited = {start_e}
    current, current_type = start_e, 'E'

    for _ in range(max_steps):
        if current_type == 'E':
            candidates = W.loc[current, inhibitory]
            valid = candidates[candidates > 0]
            next_type = 'I'
        else:
            candidates = W.loc[current, excitatory]
            valid = candidates[candidates < 0]
            next_type = 'E'

        if valid.empty:
            return chain, [], 'dead end'

        nxt = valid.abs().idxmax()   # largest absolute weight

        if nxt in visited:
            chain.append((nxt, next_type))
            return chain, [], f'cycle (would revisit {nxt})'

        visited.add(nxt)
        chain.append((nxt, next_type))
        current, current_type = nxt, next_type

    return chain, [], f'max steps ({max_steps})'


# ── Variant B: skip revisit, continue ────────────────────────
def ei_chain_varB(start_e, W, excitatory, inhibitory, max_steps=100):
    """E→I→E→I chain, skip revisit targets and continue to next unvisited.
    Records skipped revisits as side loops.
    Stops when no unvisited targets remain.
    """
    chain      = [(start_e, 'E')]
    visited    = {start_e}
    side_loops = []
    current, current_type = start_e, 'E'

    for _ in range(max_steps):
        if current_type == 'E':
            candidates = W.loc[current, inhibitory]
            valid = candidates[candidates > 0]
            next_type = 'I'
        else:
            candidates = W.loc[current, excitatory]
            valid = candidates[candidates < 0]
            next_type = 'E'

        if valid.empty:
            return chain, side_loops, 'dead end'

        # Record any would-be revisits as side loops before filtering
        revisits   = valid[[n for n in valid.index if n in visited]]
        unvisited  = valid[[n for n in valid.index if n not in visited]]

        for rv in revisits.abs().sort_values(ascending=False).index:
            side_loops.append((current, rv, valid[rv], current_type + '→' + next_type))

        if unvisited.empty:
            nxt = valid.abs().idxmax()
            return chain, side_loops, f'exhausted (no unvisited; peek: {nxt})'

        nxt = unvisited.abs().idxmax()
        visited.add(nxt)
        chain.append((nxt, next_type))
        current, current_type = nxt, next_type

    return chain, side_loops, f'max steps ({max_steps})'


# ── Variant C: edge exhaustion, revisits allowed ────────────────
def ei_chain_varC(start_e, W, excitatory, inhibitory, max_steps=500):
    """E→I→E→I chain, each directed edge used at most once; nodes revisitable.
    Stops when no unused outgoing edges remain from current node.
    """
    chain        = [(start_e, 'E')]
    used_edges   = set()
    visit_counts = {start_e: 1}
    current, current_type = start_e, 'E'

    for _ in range(max_steps):
        if current_type == 'E':
            candidates = W.loc[current, inhibitory]
            valid = candidates[candidates > 0]
            next_type = 'I'
        else:
            candidates = W.loc[current, excitatory]
            valid = candidates[candidates < 0]
            next_type = 'E'

        # Filter out already-used edges
        available = valid[[n for n in valid.index
                           if (current, n) not in used_edges]]

        if available.empty:
            return chain, visit_counts, 'edge exhausted'

        nxt = available.abs().idxmax()
        used_edges.add((current, nxt))
        visit_counts[nxt] = visit_counts.get(nxt, 0) + 1
        chain.append((nxt, next_type))
        current, current_type = nxt, next_type

    return chain, visit_counts, f'max steps ({max_steps})'


# ── Run all three variants for all 27 E starting nodes ────────────
# Build pure-E terminus lookup directly from Stage 0 path CSV
pure_e_termini = (
    df_paths[df_paths['is_terminus']]
    .set_index('path_start')['e_node']
    .to_dict()
)

chain_records = []
for start in excitatory:
    chainA, _,        reasonA = ei_chain_varA(start, W, excitatory, inhibitory)
    chainB, sl_B,     reasonB = ei_chain_varB(start, W, excitatory, inhibitory)
    chainC, vc_C,     reasonC = ei_chain_varC(start, W, excitatory, inhibitory)

    def final_e(chain):
        e_nodes = [n for n, t in chain if t == 'E']
        return e_nodes[-1] if e_nodes else start

    fe_A = final_e(chainA)
    fe_B = final_e(chainB)
    fe_C = final_e(chainC)
    pure = pure_e_termini.get(start, '')

    chain_records.append({
        'start':        start,
        'region':       region(start),
        # Variant A
        'len_A':        len(chainA),
        'final_e_A':    fe_A,
        'reason_A':     reasonA,
        'match_A':      fe_A == pure,
        # Variant B
        'len_B':        len(chainB),
        'n_loops_B':    len(sl_B),
        'final_e_B':    fe_B,
        'reason_B':     reasonB,
        'match_B':      fe_B == pure,
        # Variant C
        'len_C':        len(chainC),
        'final_e_C':    fe_C,
        'reason_C':     reasonC,
        'match_C':      fe_C == pure,
    })

chain_df = pd.DataFrame(chain_records)
chain_df.to_csv(f'{TABLE_DIR}stage1_full_ei_chains.csv', index=False)

# ── Print comparison table ──────────────────────────────────
print('Full E→I→E→I chain results — all three variants')
print(f'{"Start":<38} {"Rgn":<5} '
      f'{"LenA":>5} {"TermA":<28} {"MA":>3} '
      f'{"LenB":>5} {"TermB":<28} {"MB":>3} '
      f'{"LenC":>5} {"TermC":<28} {"MC":>3}')
print('-' * 140)
for _, row in chain_df.iterrows():
    mA = '✓' if row['match_A'] else '✗'
    mB = '✓' if row['match_B'] else '✗'
    mC = '✓' if row['match_C'] else '✗'
    tA = row['final_e_A'][:26] if len(row['final_e_A']) > 26 else row['final_e_A']
    tB = row['final_e_B'][:26] if len(row['final_e_B']) > 26 else row['final_e_B']
    tC = row['final_e_C'][:26] if len(row['final_e_C']) > 26 else row['final_e_C']
    print(f'{row["start"]:<38} {row["region"]:<5} '
          f'{int(row["len_A"]):>5} {tA:<28} {mA:>3} '
          f'{int(row["len_B"]):>5} {tB:<28} {mB:>3} '
          f'{int(row["len_C"]):>5} {tC:<28} {mC:>3}')

print(f'\nVariant A: {chain_df["match_A"].sum()}/27 match pure-E terminus')
print(f'Variant B: {chain_df["match_B"].sum()}/27 match pure-E terminus')
print(f'Variant C: {chain_df["match_C"].sum()}/27 match pure-E terminus')
print(f'\nSaved: {TABLE_DIR}stage1_full_ei_chains.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab environment detected. Drive mounted.
  Weight matrix         : ✓  (/content/drive/MyDrive/Ascoli_LabRotation/matrices/w_ij_gaa.csv)
  Neuron params         : ✓  (/content/drive/MyDrive/Ascoli_LabRotation/matrices/izhikevich_72_v3.csv)
  Stage 0 paths         : ✓  (/content/drive/MyDrive/Ascoli_LabRotation/outputs/scaffolds/stage0_variantA_paths.csv)
  Stage 0 rankings      : ✓  (/content/drive/MyDrive/Ascoli_LabRotation/outputs/scaffolds/stage0_hub_rankings.csv)
  Stage 0 EE netlist    : ✓  (/content/drive/MyDrive/Ascoli_LabRotation/outputs/scaffolds/ee_netlist.csv)

Output directory: /content/drive/MyDrive/Ascoli_LabRotation/outputs/stage1/
Full E→I→E→I chain results — all three variants
Start                                  Rgn    LenA TermA                         MA  LenB TermB                         MB  LenC TermC                         MC
-----

In [6]:
# ── Build tidy step-level DataFrames and save as CSV ─────────────────────────
# One row per step per starting node, for each variant.
# Columns allow filtering/grouping by variant, start, step, node type, region,
# edge weight, revisit status, and terminus match — suitable for external stats.

def chain_to_rows(start, chain, reason, variant, W, pure_e_termini,
                  side_loops=None, visit_counts=None):
    """Convert a single chain result to a list of row dicts (one per step)."""
    rows = []
    e_nodes = [n for n, t in chain if t == 'E']
    final_e = e_nodes[-1] if e_nodes else start
    pure    = pure_e_termini.get(start, '')
    matches_pure_e = (final_e == pure)

    # Build a set of skipped-revisit targets for Variant B annotation
    skip_targets = set()
    if side_loops:
        skip_targets = {rv for _, rv, _, _ in side_loops}

    running_counts = {}
    for step_idx, (node, ntype) in enumerate(chain):
        running_counts[node] = running_counts.get(node, 0) + 1

        # Edge weight from previous node
        if step_idx == 0:
            edge_w = None
            abs_edge_w = None
        else:
            prev = chain[step_idx - 1][0]
            edge_w = float(W.loc[prev, node])
            abs_edge_w = abs(edge_w)

        is_terminus = (step_idx == len(chain) - 1)
        visit_num   = running_counts[node]

        rows.append({
            'variant':          variant,
            'start':            start,
            'start_region':     region(start),
            'step':             step_idx,
            'node':             node,
            'node_type':        ntype,           # 'E' or 'I'
            'node_region':      region(node),
            'edge_weight':      edge_w,          # signed (negative for I->E)
            'abs_edge_weight':  abs_edge_w,      # unsigned for sorting/stats
            'visit_number':     visit_num,       # >1 means node was revisited
            'is_revisit':       visit_num > 1,
            'is_terminus':      is_terminus,
            'stop_reason':      reason if is_terminus else '',
            'final_e_node':     final_e if is_terminus else '',
            'final_e_region':   region(final_e) if is_terminus else '',
            'matches_pure_e':   matches_pure_e if is_terminus else '',
            'pure_e_terminus':  pure if is_terminus else '',
        })
    return rows


# ── Run all variants and collect step-level rows ──────────────────────────────
all_rows = []

for start in excitatory:
    chainA, _,    reasonA = ei_chain_varA(start, W, excitatory, inhibitory)
    chainB, sl_B, reasonB = ei_chain_varB(start, W, excitatory, inhibitory)
    chainC, vc_C, reasonC = ei_chain_varC(start, W, excitatory, inhibitory)

    all_rows += chain_to_rows(start, chainA, reasonA, 'A', W, pure_e_termini)
    all_rows += chain_to_rows(start, chainB, reasonB, 'B', W, pure_e_termini,
                              side_loops=sl_B)
    all_rows += chain_to_rows(start, chainC, reasonC, 'C', W, pure_e_termini,
                              visit_counts=vc_C)

steps_df = pd.DataFrame(all_rows)

# ── Save combined (all variants) and per-variant CSVs ────────────────────────
steps_df.to_csv(f'{TABLE_DIR}stage1_ei_chains_all_steps.csv', index=False)

for v in ['A', 'B', 'C']:
    sub = steps_df[steps_df['variant'] == v]
    sub.to_csv(f'{TABLE_DIR}stage1_ei_chains_variant{v}_steps.csv', index=False)

# ── Also save per-start-node summary (one row per start node per variant) ─────
# This is a convenience table: terminus, chain length, match, stop reason.
summary_rows = []
for start in excitatory:
    for v in ['A', 'B', 'C']:
        sub = steps_df[(steps_df['variant'] == v) & (steps_df['start'] == start)]
        term_row = sub[sub['is_terminus']].iloc[0] if sub['is_terminus'].any() else None
        summary_rows.append({
            'variant':          v,
            'start':            start,
            'start_region':     region(start),
            'chain_length':     len(sub),
            'n_e_nodes':        (sub['node_type'] == 'E').sum(),
            'n_i_nodes':        (sub['node_type'] == 'I').sum(),
            'n_revisits':       sub['is_revisit'].sum(),
            'final_e_node':     term_row['final_e_node']    if term_row is not None else '',
            'final_e_region':   term_row['final_e_region']  if term_row is not None else '',
            'matches_pure_e':   term_row['matches_pure_e']  if term_row is not None else '',
            'pure_e_terminus':  term_row['pure_e_terminus'] if term_row is not None else '',
            'stop_reason':      term_row['stop_reason']     if term_row is not None else '',
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f'{TABLE_DIR}stage1_ei_chains_summary.csv', index=False)

# ── Report ───────────────────────────────────────────────────────────────────
print('CSV outputs saved:')
print(f'  stage1_ei_chains_all_steps.csv       — {len(steps_df):>5} rows  (all variants, all steps)')
for v in ['A', 'B', 'C']:
    n = len(steps_df[steps_df['variant'] == v])
    print(f'  stage1_ei_chains_variantA_steps.csv  — {n:>5} rows  (Variant {v} step-level)'
          .replace('variantA', f'variant{v}'))
print(f'  stage1_ei_chains_summary.csv         — {len(summary_df):>5} rows  (one per start×variant)')
print()
print('Columns in step-level CSV:')
for col in steps_df.columns:
    print(f'  {col}')


CSV outputs saved:
  stage1_ei_chains_all_steps.csv       —   511 rows  (all variants, all steps)
  stage1_ei_chains_variantA_steps.csv  —   102 rows  (Variant A step-level)
  stage1_ei_chains_variantB_steps.csv  —   130 rows  (Variant B step-level)
  stage1_ei_chains_variantC_steps.csv  —   279 rows  (Variant C step-level)
  stage1_ei_chains_summary.csv         —    84 rows  (one per start×variant)

Columns in step-level CSV:
  variant
  start
  start_region
  step
  node
  node_type
  node_region
  edge_weight
  abs_edge_weight
  visit_number
  is_revisit
  is_terminus
  stop_reason
  final_e_node
  final_e_region
  matches_pure_e
  pure_e_terminus


In [7]:
# ── Variant B: full path printouts for all 27 starting nodes ─────────────────
KEY_STARTS_B = excitatory   # print all 27; narrow to a subset if preferred

for start in KEY_STARTS_B:
    chainB, sl_B, reasonB = ei_chain_varB(start, W, excitatory, inhibitory)
    e_nodes_B = [n for n, t in chainB if t == 'E']
    print('=' * 100)
    print(f'VARIANT B PATH FROM: {start}')
    print(f'Chain length: {len(chainB)} nodes | E nodes: {len(e_nodes_B)} | '
          f'Side loops skipped: {len(sl_B)} | Stop: {reasonB}')
    print()
    print(f'  {"Step":<5} {"Node":<42} {"Type":<4} {"->|w|":>12}  Skipped revisit(s)')
    print('  ' + '-' * 95)

    # Build skip map: step -> list of (skipped_node, weight, direction)
    visited_so_far = set()
    skip_map = {}
    for step_idx, (node, ntype) in enumerate(chainB):
        visited_so_far.add(node)
        if step_idx < len(chainB) - 1:
            if ntype == 'E':
                cands = W.loc[node, inhibitory]
                valid = cands[cands > 0]
            else:
                cands = W.loc[node, excitatory]
                valid = cands[cands < 0]
            skips = [(n, valid[n]) for n in valid.index if n in visited_so_far
                     and n != chainB[step_idx+1][0]]
            if skips:
                skip_map[step_idx] = sorted(skips, key=lambda x: abs(x[1]), reverse=True)

    for step_idx, (node, ntype) in enumerate(chainB):
        if step_idx == 0:
            w_str = '(start)'
        else:
            prev_node = chainB[step_idx-1][0]
            w_val = W.loc[prev_node, node]
            w_str = f'+[{abs(w_val):.2e}]'

        skips = skip_map.get(step_idx, [])
        if skips:
            first = skips[0]
            skip_str = f'~> {first[0][:35]} [{first[1]:.2e}]'
            print(f'  {step_idx:<5} {node:<42} {ntype:<4} {w_str:>12}  {skip_str}')
            for sk_node, sk_w in skips[1:]:
                print(f'  {"":5} {"":42} {"":4} {"":12}  ~> {sk_node[:35]} [{sk_w:.2e}]')
        else:
            print(f'  {step_idx:<5} {node:<42} {ntype:<4} {w_str:>12}')

        if step_idx == len(chainB) - 1:
            fe = e_nodes_B[-1] if e_nodes_B else start
            pure = pure_e_termini.get(start, 'unknown')
            match = '✓ matches pure-E terminus' if fe == pure else f'✗ rerouted (pure-E: {pure})'
            print(f'  -> TERMINUS: {reasonB}')
            print(f'     Final E node: {fe}  [{region(fe)}]  {match}')
    print()


VARIANT B PATH FROM: CA1 Pyramidal
Chain length: 4 nodes | E nodes: 2 | Side loops skipped: 4 | Stop: exhausted (no unvisited; peek: CA1 Pyramidal)

  Step  Node                                       Type        ->|w|  Skipped revisit(s)
  -----------------------------------------------------------------------------------------------
  0     CA1 Pyramidal                              E         (start)
  1     CA1 Recurrent O LM                         I     +[1.73e+05]  ~> CA1 Pyramidal [-1.77e+02]
  2     CA1 Radiatum Giant                         E     +[1.35e+02]  ~> CA1 Recurrent O LM [2.62e+03]
  3     CA1 Trilaminar                             I     +[2.39e+03]
  -> TERMINUS: exhausted (no unvisited; peek: CA1 Pyramidal)
     Final E node: CA1 Radiatum Giant  [CA1]  ✓ matches pure-E terminus

VARIANT B PATH FROM: CA1 Radiatum Giant
Chain length: 4 nodes | E nodes: 2 | Side loops skipped: 4 | Stop: exhausted (no unvisited; peek: CA1 Pyramidal)

  Step  Node                        

In [8]:
# ── Variant C: full path printouts for all 27 starting nodes ─────────────────
# Variant C revisits nodes freely, so a linear path list can be very long.
# We print step, node, type, and edge weight. Visit count is shown in brackets
# for any node visited more than once.

KEY_STARTS_C = excitatory

for start in KEY_STARTS_C:
    chainC, vc_C, reasonC = ei_chain_varC(start, W, excitatory, inhibitory)
    e_nodes_C = [n for n, t in chainC if t == 'E']
    print('=' * 100)
    print(f'VARIANT C PATH FROM: {start}')
    print(f'Chain length: {len(chainC)} nodes | E nodes: {len(e_nodes_C)} | Stop: {reasonC}')
    print()
    print(f'  {"Step":<5} {"Node":<42} {"Type":<4} {"->|w|":>12}  [visit#]')
    print('  ' + '-' * 75)

    running_counts = {}
    for step_idx, (node, ntype) in enumerate(chainC):
        running_counts[node] = running_counts.get(node, 0) + 1
        visit_label = f'[#{running_counts[node]}]' if running_counts[node] > 1 else ''

        if step_idx == 0:
            w_str = '(start)'
        else:
            prev_node = chainC[step_idx-1][0]
            w_val = W.loc[prev_node, node]
            w_str = f'+[{abs(w_val):.2e}]'

        print(f'  {step_idx:<5} {node:<42} {ntype:<4} {w_str:>12}  {visit_label}')

    fe = e_nodes_C[-1] if e_nodes_C else start
    pure = pure_e_termini.get(start, 'unknown')
    match = '✓ matches pure-E terminus' if fe == pure else f'✗ rerouted (pure-E: {pure})'
    print(f'  -> TERMINUS: {reasonC}')
    print(f'     Final E node: {fe}  [{region(fe)}]  {match}')
    print()


VARIANT C PATH FROM: CA1 Pyramidal
Chain length: 14 nodes | E nodes: 7 | Stop: edge exhausted

  Step  Node                                       Type        ->|w|  [visit#]
  ---------------------------------------------------------------------------
  0     CA1 Pyramidal                              E         (start)  
  1     CA1 Recurrent O LM                         I     +[1.73e+05]  
  2     CA1 Pyramidal                              E     +[1.77e+02]  [#2]
  3     CA1 Trilaminar                             I     +[1.33e+05]  
  4     CA1 Pyramidal                              E     +[1.39e+03]  [#3]
  5     CA1 Horizontal Basket                      I     +[1.11e+05]  
  6     CA1 Radiatum Giant                         E     +[4.46e+03]  
  7     CA1 Recurrent O LM                         I     +[2.62e+03]  [#2]
  8     CA1 Radiatum Giant                         E     +[1.35e+02]  [#2]
  9     CA1 Trilaminar                             I     +[2.39e+03]  [#2]
  10    CA1 Radiat

In [9]:
# ── Terminus distribution across all three variants ─────────────────────────
print('Terminus distribution — Variant A (final E node):')
for node, cnt in chain_df['final_e_A'].value_counts().items():
    sym = '✓' if node == 'MEC LV VI Pyramidal Polymorphic' else ' '
    print(f'  {sym} {cnt:2d}/27  {node}  [{region(node)}]')

print('\nTerminus distribution — Variant B (final E node):')
for node, cnt in chain_df['final_e_B'].value_counts().items():
    sym = '✓' if node == 'MEC LV VI Pyramidal Polymorphic' else ' '
    print(f'  {sym} {cnt:2d}/27  {node}  [{region(node)}]')

print('\nTerminus distribution — Variant C (final E node):')
for node, cnt in chain_df['final_e_C'].value_counts().items():
    sym = '✓' if node == 'MEC LV VI Pyramidal Polymorphic' else ' '
    print(f'  {sym} {cnt:2d}/27  {node}  [{region(node)}]')

print('\n(✓ = same as pure excitatory greedy terminus: MEC LV VI Pyramidal Polymorphic)')


Terminus distribution — Variant A (final E node):
     9/27  EC LI II Multipolar Pyramidal  [EC]
     7/27  DG Mossy  [DG]
     6/27  CA1 Pyramidal  [CA1]
     1/27  EC LIV VI Deep Multipolar Principal  [EC]
     1/27  EC LV Deep Pyramidal  [EC]
     1/27  LEC LIII Complex Pyramidal  [LEC]
     1/27  LEC LIII Multipolar Principal  [LEC]
     1/27  LEC LVI Multipolar Pyramidal  [LEC]
  ✓  1/27  MEC LV VI Pyramidal Polymorphic  [MEC]

Terminus distribution — Variant B (final E node):
     8/27  CA1 Radiatum Giant  [CA1]
     8/27  EC LI II Multipolar Pyramidal  [EC]
     3/27  CA1 Pyramidal  [CA1]
     2/27  DG Granule  [DG]
     1/27  DG Mossy  [DG]
     1/27  EC LIV VI Deep Multipolar Principal  [EC]
     1/27  EC LV Deep Pyramidal  [EC]
     1/27  LEC LIII Complex Pyramidal  [LEC]
     1/27  LEC LIII Multipolar Principal  [LEC]
     1/27  LEC LVI Multipolar Pyramidal  [LEC]
  ✓  1/27  MEC LV VI Pyramidal Polymorphic  [MEC]

Terminus distribution — Variant C (final E node):
    17/27  

## 4. Summary

| Component | Key Finding |
|-----------|-------------|
| E→I→E sidechain | For each step on canonical path: strongest E→I target and its suppression target annotated as parallel columns |
| E→I→E→I Variant A | Stop on revisit — shortest chains, conservative terminus |
| E→I→E→I Variant B | Skip revisit, continue — longer chains with side loops recorded; full path printed |
| E→I→E→I Variant C | Edge exhaustion — most exhaustive; nodes revisitable, edges each used once; full path printed |
| Terminus convergence | Do all three inhibitory chain variants converge to same attractor as pure-E greedy path? |

**Bug fixed:** Suppression target lookup restricted to excitatory nodes only

**Open questions / next steps:**
- I→I disinhibition (inhibition of inhibition) — Stage 1b, deferred
- CA3 bifurcation analysis (Wilson-Cowan sweep on CA3 recurrent block) — after Stage 1 is finalized
- Integration of K_j vector from Jeanne to compute full M_ij = W_ij × K_j

**Next:** Stage 2 — Weighted Excitability Score (wES) trimer scoring.

In [10]:
import pandas as pd

# ---------------------------------------------------
# 1. Stage 0 termini
# Requires df_paths from stage0_variantA_paths.csv
# ---------------------------------------------------
stage0 = (
    df_paths[df_paths['is_terminus']]
    [['path_start', 'e_node']]
    .rename(columns={'e_node':'stage0_end'})
)

# ---------------------------------------------------
# 2. Stage 1 termini
# Requires compare_df or variant outputs from stage1
# Assumes columns:
# start, final_e_A, final_e_B, final_e_C
# ---------------------------------------------------
stage1 = compare_df.rename(columns={
    'start':'path_start',
    'final_e_A':'stage1_A_end',
    'final_e_B':'stage1_B_end',
    'final_e_C':'stage1_C_end'
})

# ---------------------------------------------------
# 3. Merge
# ---------------------------------------------------
merged = stage0.merge(stage1, on='path_start', how='inner')

# ---------------------------------------------------
# 4. Exact matches
# ---------------------------------------------------
for col in ['stage1_A_end','stage1_B_end','stage1_C_end']:
    merged[col.replace('_end','_match')] = merged[col] == merged['stage0_end']

# ---------------------------------------------------
# 5. Region helper
# ---------------------------------------------------
def region_of(x):
    if pd.isna(x): return None
    for r in ['DG','CA3','CA2','CA1','MEC','LEC','EC']:
        if r in str(x):
            return r
    return 'Other'

merged['stage0_region'] = merged['stage0_end'].apply(region_of)

for col in ['stage1_A_end','stage1_B_end','stage1_C_end']:
    rcol = col.replace('_end','_region')
    mcol = col.replace('_end','_region_match')
    merged[rcol] = merged[col].apply(region_of)
    merged[mcol] = merged[rcol] == merged['stage0_region']

# ---------------------------------------------------
# 6. Summary
# ---------------------------------------------------
summary = []

for v in ['A','B','C']:
    exact = merged[f'stage1_{v}_match'].sum()
    region = merged[f'stage1_{v}_region_match'].sum()
    total = len(merged)

    summary.append({
        'Variant': v,
        'Exact node matches': f'{exact}/{total}',
        'Region matches': f'{region}/{total}',
        'Exact %': round(100*exact/total,1),
        'Region %': round(100*region/total,1)
    })

summary_df = pd.DataFrame(summary)
print(summary_df)

# ---------------------------------------------------
# 7. Show reroutes
# ---------------------------------------------------
reroutes = merged[
    ~(merged['stage1_A_match'] &
      merged['stage1_B_match'] &
      merged['stage1_C_match'])
]

display(reroutes[['path_start','stage0_end',
                  'stage1_A_end','stage1_B_end','stage1_C_end']])

NameError: name 'compare_df' is not defined